# Publication venues
Sections 4.3.1-4.3.2 and Table 2. Analysis starts from stored Crossref metadata,
an OpenAlex source snapshot, saved journal issue counts and a selection manifest.
No network requests are made. The manually researched fee/review-time section is
outside this notebook.

Default paths are relative: `data/` holds the original data, `data/venues/`
holds the saved snapshot and selection manifest, and
`outputs/09_publication_venues/` holds aggregate results.

Required input schemas:
- `{pred,norm}/dataset/csv/papers.csv`: `id`, `publisher` (observed venue name).
- `pred/metadata/pred_crossref_meta_clean.csv` and
  `norm/metadata/norm_crossref_meta.csv`: `id`, `issn`.
- `openalex_source_snapshot.json`: `retrieved_at_utc`, `records_by_issn` mapping
  ISSNs to source records with `id`, `display_name`, `type`, `cited_by_count`,
  `works_count`, `h_index`, `i10_index`, `mean_2yr_citedness`, `is_oa`, `is_core`.
- `journal_growth_selection.csv`: `Group`, `Selection order`, `Journal`;
  groups `Predatory` and `Normal`, four manually specified selections each.
- `pred/metadata/pred_journals_count.xlsx` and
  `norm/metadata/good_journals_count.xlsx`, sheet `Sheet1`:
  `Journal`, `year`, `month`, `volume`, `Issue`, `paper count`.

The Predatory metadata input is the existing cleaned one-row-per-paper table.
Venue names, ISSNs and source records are loaded from data files, never
embedded in notebook code. Inputs are not supplied with the code.

In [ ]:
import os
import json
import re
import unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

REPO_ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
DATA_DIR = REPO_ROOT / 'data'
VENUE_INPUT_DIR = DATA_DIR / 'venues'
OUTPUT_DIR = REPO_ROOT / 'outputs' / '09_publication_venues'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
GROUPS={"Predatory":"pred","Normal":"norm"}

def save_table(frame,name):
    frame.to_csv(OUTPUT_DIR/(name+".csv"),index=False)
    display(frame)

with (VENUE_INPUT_DIR/"openalex_source_snapshot.json").open(encoding="utf-8") as stream:
    snapshot=json.load(stream)
openalex_by_issn=snapshot["records_by_issn"]
print("Stored OpenAlex snapshot date:",snapshot["retrieved_at_utc"])


## Journal cohorts
Use all observed Google Scholar publication venues, not only cited-paper venues.
Normalize venue names with Unicode NFKC, case folding and retention of alphanumeric
characters. Missing venues include nulls, blanks and the literal `null`.
Join stored paper-level ISSNs one-to-one by the dataset-local paper key.
ISSN syntax is checked without adding a checksum rule. An enriched source must
have type `journal` and a normalized display name exactly matching the observed
venue. Deduplicate by OpenAlex Source ID, then exclude sources shared by both sets
from both comparison groups.

In [ ]:
def normalize_venue(value):
    text = unicodedata.normalize("NFKC", str(value)).casefold()
    return "".join(character for character in text if character.isalnum())

def valid_venue_mask(series):
    text = series.astype("string")
    return (
        text.notna()
        & text.str.strip().ne("")
        & ~text.str.strip().str.lower().eq("null")
    )

def split_valid_issns(value):
    if pd.isna(value):
        return []
    return [
        candidate.strip().upper()
        for candidate in str(value).split(",")
        if re.fullmatch(
            r"\d{4}-\d{3}[\dXx]", candidate.strip()
        )
    ]

def observed_sources(group):
    folder=DATA_DIR/GROUPS[group]
    papers=pd.read_csv(folder/"dataset"/"csv"/"papers.csv",dtype=str,usecols=["id","publisher"])
    metadata_name="pred_crossref_meta_clean.csv" if group=="Predatory" else "norm_crossref_meta.csv"
    metadata=pd.read_csv(folder/"metadata"/metadata_name,dtype=str,usecols=["id","issn"])
    for frame in [papers,metadata]:
        if frame.id.isna().any() or frame.id.duplicated().any():
            raise ValueError("Paper and metadata keys must be nonmissing and unique.")
    joined=papers.merge(metadata,on="id",how="left",validate="one_to_one")
    venues=joined.loc[valid_venue_mask(joined.publisher)].copy()
    venues["normalized_venue"]=venues.publisher.str.strip().map(normalize_venue)
    venue_to_issns={}
    for row in venues.itertuples(index=False):
        venue_to_issns.setdefault(row.normalized_venue,set()).update(split_valid_issns(row.issn))
    sources={}
    for venue,issns in venue_to_issns.items():
        for issn in sorted(issns):
            source=openalex_by_issn.get(issn)
            if source and source.get("type")=="journal" and normalize_venue(source.get("display_name",""))==venue:
                sources[source["id"]]=source
    return sources

observed={label:observed_sources(label) for label in GROUPS}
shared=set(observed["Predatory"]) & set(observed["Normal"])
final_ids={label:set(sources)-shared for label,sources in observed.items()}
assert not final_ids["Predatory"] & final_ids["Normal"]
save_table(pd.DataFrame([{"Group":label,"Observed journal sources":len(observed[label]),
                          "Shared sources excluded":len(shared),"Group-specific journals":len(final_ids[label])}
                         for label in GROUPS]),"journal_cohort_counts")


## Journal metrics (Table 2)
Five numeric outcomes use two-sided asymptotic Mann-Whitney U tests and signed
Cliff's delta, `2*U/(n_P*n_N)-1`. Two binary outcomes use two-sided Fisher exact
tests and the Predatory-versus-Normal odds ratio. Exclude missing values separately
for each metric. Apply Benjamini-Hochberg correction jointly to all seven p-values.
Two-year mean citedness is a ratio, not a percentage of papers receiving citations.

In [ ]:
def cliffs_delta_from_u(u, n_pred, n_norm):
    return 2 * u / (n_pred * n_norm) - 1

def benjamini_hochberg(p_values):
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    ranked = p[order]
    adjusted = np.empty_like(ranked)
    running = 1.0
    m = len(p)
    for i in range(m - 1, -1, -1):
        running = min(running, ranked[i] * m / (i + 1))
        adjusted[i] = running
    result = np.empty_like(adjusted)
    result[order] = adjusted
    return result

metrics={label:pd.DataFrame([observed[label][key] for key in sorted(final_ids[label])]) for label in GROUPS}
continuous_rows=[]
for column,title in {"cited_by_count":"Cited by count","works_count":"Works count",
                      "h_index":"h-index","i10_index":"i10-index","mean_2yr_citedness":"2-yr citedness"}.items():
    p,n=[pd.to_numeric(metrics[label][column],errors="coerce").dropna() for label in GROUPS]
    assert np.isfinite(p).all() and np.isfinite(n).all()
    u,pv=stats.mannwhitneyu(p,n,alternative="two-sided",method="asymptotic")
    continuous_rows.append({"Metric":title,"Predatory n":len(p),"Predatory mean":p.mean(),"Predatory median":p.median(),
                            "Normal n":len(n),"Normal mean":n.mean(),"Normal median":n.median(),
                            "U statistic":u,"Cliff's delta":cliffs_delta_from_u(u,len(p),len(n)),"Raw p-value":pv})
binary_rows=[]
for column,title in {"is_oa":"Open access","is_core":"Leiden core"}.items():
    values=[metrics[label][column].dropna() for label in GROUPS]
    if any(not x.isin([True,False]).all() for x in values):
        raise ValueError("Stored binary metrics must be booleans or binary values.")
    p,n=[x.astype(bool) for x in values]
    table=np.array([[p.sum(),(~p).sum()],[n.sum(),(~n).sum()]],dtype=int)
    odds,pv=stats.fisher_exact(table,alternative="two-sided")
    binary_rows.append({"Metric":title,"Predatory n":len(p),"Predatory true":int(p.sum()),"Predatory percent":100*p.mean(),
                        "Normal n":len(n),"Normal true":int(n.sum()),"Normal percent":100*n.mean(),
                        "Odds ratio":odds,"Raw p-value":pv})
rows=continuous_rows+binary_rows
for row,q in zip(rows,benjamini_hochberg([row["Raw p-value"] for row in rows])):
    row["BH q-value"]=q
save_table(pd.DataFrame(continuous_rows),"table2_numeric_metrics")
save_table(pd.DataFrame(binary_rows),"table2_binary_metrics")


## Selected venue growth, 2022-2024
Read the four selections per set from the external manifest, preserving their
specified order. The source analysis specifies these lists explicitly; it does
not derive them through an automatic volume ranking. Selection provenance is
therefore separate from the annual count calculation.

Sum stored issue-level paper counts within each selected journal and calendar
year. Strip whitespace from spreadsheet column names. An all-missing group stays
missing (`min_count=1`); require observed nonmissing counts for every selected
journal in each of the three years. Fold change is the 2024 count divided by the
2022 count. No inferential test, alternative scale or exploratory ranking is added.

In [ ]:
selection=pd.read_csv(VENUE_INPUT_DIR/"journal_growth_selection.csv",usecols=["Group","Selection order","Journal"])
if set(selection.Group)!=set(GROUPS) or selection[["Group","Journal"]].isna().any().any():
    raise ValueError("Selection manifest must contain both groups and nonmissing journal names.")
if selection.duplicated(["Group","Journal"]).any() or selection.duplicated(["Group","Selection order"]).any():
    raise ValueError("Selections and their ordering must be unique within groups.")
if not selection.groupby("Group").size().eq(4).all():
    raise ValueError("Exactly four selected journals per group are required.")
growth_rows=[]
for label,group in GROUPS.items():
    filename="pred_journals_count.xlsx" if label=="Predatory" else "good_journals_count.xlsx"
    raw=pd.read_excel(DATA_DIR/group/"metadata"/filename,sheet_name="Sheet1")
    raw.columns=[str(c).strip() for c in raw.columns]
    required={"Journal","year","month","volume","Issue","paper count"}
    if not required.issubset(raw.columns):
        raise ValueError("Issue-count spreadsheet is missing required columns.")
    raw["year"]=pd.to_numeric(raw["year"],errors="coerce").astype("Int64")
    raw["paper count"]=pd.to_numeric(raw["paper count"],errors="coerce")
    selected=selection.loc[selection.Group.eq(label)].sort_values("Selection order")
    annual=raw.loc[raw.Journal.isin(selected.Journal)].groupby(["Journal","year"])["paper count"].sum(min_count=1)
    for row in selected.itertuples(index=False,name=None):
        _,order,journal=row
        counts=[annual.get((journal,year),np.nan) for year in [2022,2023,2024]]
        if not np.isfinite(counts).all() or counts[0]<=0:
            raise ValueError("Every selected journal needs finite annual counts and a positive 2022 baseline.")
        growth_rows.append({"Group":label,"Selection order":order,"2022":counts[0],"2023":counts[1],"2024":counts[2],
                            "Fold change":counts[2]/counts[0]})
save_table(pd.DataFrame(growth_rows),"selected_venue_growth")
